# Attach downsampled DAPI to the AnnData objects

Adds a downsampled DAPI thumbnail per region into each `.h5ad` using the standard scanpy/squidpy spatial
convention, so the tissue image can be shown under the cells in a spatial viewer (KaroSpace / `sc.pl.spatial`).

For each sample we:
1. stream-downsample the (huge) MERSCOPE DAPI mosaic to a small thumbnail (bounded memory, **no GPU**),
2. **register** it to the Baysor cell centroids (which are in microns) by fitting the cell-centroid bounding
   box to the image — and validate that the implied pixel size is ~0.108 µm/px,
3. store it in `adata.uns['spatial'][sample]['images']['hires']` with a scalefactor, and add a pixel-space
   copy of the coordinates in `adata.obsm['spatial_px']` (the micron `obsm['spatial']` is left untouched).

> **Caveat — this is a visualization-grade registration, not exact.** Without the MERSCOPE
> `micron_to_mosaic_pixel_transform.csv` we align bounding boxes, which absorbs the few-percent tissue margin
> as a small anisotropic stretch (~3%). Good enough to eyeball image vs. clusters; for transcript-accurate
> overlay (e.g. a Baysor prior) we'll still want the real transform.

**Kernel:** `Python (sopa)` (has tifffile + zarr + anndata).

In [ ]:
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import tifffile, zarr
import anndata as ad
import matplotlib.pyplot as plt

DAPI_ROOT = Path("/Users/christoffer/Downloads/dapi")
H5_DIR    = Path("/Volumes/T7/Stroke_merscop_Fan_CG/h5ad")

PIXEL_SIZE_UM = 0.108     # expected MERSCOPE mosaic pixel size (for the validation check)
THUMB_MAX_PX  = 2000      # longest side of the stored thumbnail
FLIP_Y        = False     # set True if the overlay check shows the image upside-down vs cells

tif_index = {Path(f).stem: f for f in glob.glob(str(DAPI_ROOT / "*" / "*.tif"))}
print(f"{len(tif_index)} DAPI tifs available")
print(f"{len(list(H5_DIR.glob('*.h5ad')))} h5ad files in {H5_DIR}")

## 1. Helpers

In [ ]:
def read_thumbnail(path, target_max_px, band_rows=8192):
    """Stream-downsample a strip TIFF to a small uint8 thumbnail. Returns (thumb, f, H, W)."""
    store = tifffile.imread(path, aszarr=True)
    try:
        z = zarr.open(store, mode="r")
        H, W = z.shape
        f = max(1, round(max(H, W) / target_max_px))
        Wc = (W // f) * f
        rows = []
        step = (band_rows // f) * f
        for y0 in range(0, (H // f) * f, step):
            y1 = min(y0 + step, (H // f) * f)
            band = np.asarray(z[y0:y1, :Wc]).astype(np.float32)
            r = (y1 - y0) // f
            rows.append(band[: r * f].reshape(r, f, Wc // f, f).mean(axis=(1, 3)))
        thumb = np.concatenate(rows, axis=0)
    finally:
        store.close()
    # contrast stretch to uint8 (1-99 percentile)
    lo, hi = np.percentile(thumb, [1, 99])
    thumb = np.clip((thumb - lo) / max(hi - lo, 1e-6), 0, 1)
    return (thumb * 255).astype(np.uint8), f, H, W


def register_to_image(adata, H, W, flip_y=False):
    """Map micron centroids (obsm['spatial']) to full-res image pixels by fitting bbox->image.
    Returns (spatial_px, info) where spatial_px is in FULL-RES pixel coords."""
    um = np.asarray(adata.obsm["spatial"], dtype=float)
    xmin, ymin = um.min(0)
    xmax, ymax = um.max(0)
    sx = W / (xmax - xmin)      # px per micron, x
    sy = H / (ymax - ymin)      # px per micron, y
    px_x = (um[:, 0] - xmin) * sx
    px_y = (um[:, 1] - ymin) * sy
    if flip_y:
        px_y = H - px_y
    spatial_px = np.column_stack([px_x, px_y])
    info = dict(implied_um_per_px_x=1 / sx, implied_um_per_px_y=1 / sy,
                img_H=H, img_W=W, n=adata.n_obs)
    return spatial_px, info


def attach_dapi(adata, sample, thumb, f, spatial_px):
    """Store thumbnail + scalefactor in uns['spatial'][sample]; pixel coords in obsm['spatial_px']."""
    adata.uns.setdefault("spatial", {})
    adata.uns["spatial"][sample] = {
        "images": {"hires": thumb},
        "scalefactors": {
            "tissue_hires_scalef": 1.0 / f,   # full-res px * scalef -> thumbnail px
            "spot_diameter_fullres": 10.0 / PIXEL_SIZE_UM,
        },
    }
    adata.obsm["spatial_px"] = spatial_px

## 2. Process each per-sample h5ad

Loads each `<sample>.h5ad`, attaches its DAPI thumbnail, validates the implied pixel size, and saves in place.

In [ ]:
h5_files = sorted(p for p in H5_DIR.glob("*.h5ad") if not p.name.startswith("._") and p.stem != "stroke_all")

report = []
thumbs = {}   # keep for the concatenated object + the overlay check
for p in h5_files:
    sample = p.stem
    if sample not in tif_index:
        print(f"!! no DAPI tif for {sample} — skipping")
        continue
    a = ad.read_h5ad(p)
    thumb, f, H, W = read_thumbnail(tif_index[sample], THUMB_MAX_PX)
    spatial_px, info = register_to_image(a, H, W, flip_y=FLIP_Y)
    attach_dapi(a, sample, thumb, f, spatial_px)
    a.write_h5ad(p)
    thumbs[sample] = (thumb, f, spatial_px)
    report.append(dict(sample=sample, n=info["n"], thumb=f"{thumb.shape[1]}x{thumb.shape[0]}",
                       ds=f, um_px_x=round(info["implied_um_per_px_x"], 4),
                       um_px_y=round(info["implied_um_per_px_y"], 4)))
    print(f"attached {sample}: thumb {thumb.shape[1]}x{thumb.shape[0]} (ds={f}), "
          f"implied um/px = {info['implied_um_per_px_x']:.4f},{info['implied_um_per_px_y']:.4f}")

pd.DataFrame(report)

## 3. Overlay check — are cells sitting on the tissue?

If the points look rotated/flipped relative to the DAPI, set `FLIP_Y = True` (or we may need a transpose) and
re-run section 2. The implied µm/px in the table above should be ~0.10–0.11; if it's wildly off, the bbox
assumption broke for that sample.

In [ ]:
n = len(thumbs)
if n:
    fig, axs = plt.subplots(1, n, figsize=(6 * n, 6), squeeze=False)
    for ax, (sample, (thumb, f, spx)) in zip(axs.ravel(), thumbs.items()):
        ax.imshow(thumb, cmap="gray")
        ax.scatter(spx[:, 0] / f, spx[:, 1] / f, s=0.3, c="red", alpha=0.3, linewidths=0)
        ax.set_title(sample, fontsize=9); ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("nothing attached yet")

## 4. Update the concatenated `stroke_all.h5ad`

Stores every sample's thumbnail under `uns['spatial'][sample]` and writes the combined pixel coordinates so
the whole object carries its images too.

In [ ]:
combined_path = H5_DIR / "stroke_all.h5ad"
if combined_path.exists() and thumbs:
    comb = ad.read_h5ad(combined_path)
    comb.uns.setdefault("spatial", {})
    spatial_px = np.full((comb.n_obs, 2), np.nan)
    for sample, (thumb, f, spx) in thumbs.items():
        comb.uns["spatial"][sample] = {
            "images": {"hires": thumb},
            "scalefactors": {"tissue_hires_scalef": 1.0 / f,
                              "spot_diameter_fullres": 10.0 / PIXEL_SIZE_UM},
        }
        mask = (comb.obs["sample"].astype(str) == sample).values
        spatial_px[mask] = spx
    comb.obsm["spatial_px"] = spatial_px
    comb.write_h5ad(combined_path)
    print(f"updated {combined_path.name} with {len(thumbs)} DAPI images")
    print("uns['spatial'] keys:", list(comb.uns["spatial"].keys()))
else:
    print("stroke_all.h5ad not found or no thumbnails — skipped")